# Notebook 05 — ReAct do Zero (sem Framework)

## O que e ReAct?

**ReAct** (Reasoning + Acting) e um padrao de agente em que o LLM alterna entre dois modos:

- **Reasoning** — o modelo *pensa* sobre o problema e decide o que fazer a seguir.
- **Acting** — o modelo *executa* uma acao (chamar uma tool) e observa o resultado.

O loop continua ate que o modelo decida que tem informacao suficiente para dar a resposta final ao usuario.

```
pensar → decidir tool → executar → observar → pensar → ... → responder
```

Frameworks como LangGraph implementam esse loop por baixo dos panos. Neste notebook vamos construi-lo manualmente, linha por linha, usando apenas o SDK `google-genai` diretamente — sem nenhuma abstracao de framework.

## Pseudocodigo do Loop

```
messages = [system_prompt, user_message]
while not done:
    response = llm.generate(messages, tools=tool_declarations)
    if response has tool_calls:
        for tool_call in response.tool_calls:
            result = execute_tool(tool_call)
            messages.append(tool_result)
    else:
        return response.text  # resposta final
```

Cada iteracao acrescenta mensagens ao historico de conversa. O LLM sempre recebe o historico completo — e assim que ele ve o resultado das tools anteriores e pode raciocinar sobre eles.

In [ ]:
# Setup: instalar dependencias e configurar cliente
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "google-genai>=1.0.0", "rich>=14.0.0"])

import os
from google import genai
from google.genai import types
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich import print as rprint

console = Console()

# Configurar chave de API
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "Variavel de ambiente GEMINI_API_KEY nao encontrada. "
        "Execute: export GEMINI_API_KEY='sua-chave-aqui'"
    )

client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

console.print(Panel(f"[green]Cliente configurado[/green] | Modelo: [bold]{MODEL}[/bold]",
                    title="Setup", border_style="blue"))

## Definindo as Tools

Vamos criar duas tools que o agente podera usar:

1. **`search_database`** — executa consultas SQL SELECT num banco SQLite em memoria com dados de vendas.
2. **`calculate`** — avalia expressoes matematicas de forma segura usando o modulo `ast` (sem chamar `builtins` diretamente).

In [ ]:
import sqlite3
import ast
import operator
import json

# ---------------------------------------------------------------------------
# Banco de dados SQLite em memoria com dados de vendas
# ---------------------------------------------------------------------------

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE vendas (
        id       INTEGER PRIMARY KEY,
        produto  TEXT,
        regiao   TEXT,
        trimestre TEXT,
        ano      INTEGER,
        receita  REAL,
        unidades INTEGER
    )
""")

dados = [
    (1,  "Software",  "Sul",     "Q1", 2024, 120000.00, 40),
    (2,  "Software",  "Sul",     "Q2", 2024, 145000.00, 48),
    (3,  "Software",  "Sul",     "Q3", 2024, 138000.00, 46),
    (4,  "Software",  "Sul",     "Q4", 2024, 160000.00, 53),
    (5,  "Hardware",  "Norte",   "Q1", 2024,  95000.00, 190),
    (6,  "Hardware",  "Norte",   "Q2", 2024, 102000.00, 204),
    (7,  "Hardware",  "Norte",   "Q3", 2024,  98000.00, 196),
    (8,  "Hardware",  "Norte",   "Q4", 2024, 115000.00, 230),
    (9,  "Servicos",  "Centro",  "Q1", 2024,  75000.00, 150),
    (10, "Servicos",  "Centro",  "Q2", 2024,  88000.00, 176),
    (11, "Servicos",  "Centro",  "Q3", 2024,  91000.00, 182),
    (12, "Servicos",  "Centro",  "Q4", 2024,  99000.00, 198),
    (13, "Software",  "Norte",   "Q2", 2024, 131000.00, 43),
    (14, "Hardware",  "Sul",     "Q3", 2024,  87000.00, 174),
    (15, "Servicos",  "Norte",   "Q2", 2024,  62000.00, 124),
]

cursor.executemany(
    "INSERT INTO vendas VALUES (?, ?, ?, ?, ?, ?, ?)",
    dados
)
conn.commit()

console.print(f"[green]Banco criado[/green] com [bold]{len(dados)}[/bold] registros na tabela [bold]vendas[/bold]")
console.print("Colunas: id, produto, regiao, trimestre, ano, receita, unidades")


# ---------------------------------------------------------------------------
# Implementacao das tools
# ---------------------------------------------------------------------------

def search_database(query: str) -> str:
    """Executa uma consulta SQL SELECT no banco de vendas."""
    query = query.strip()
    if not query.upper().startswith("SELECT"):
        return json.dumps({"error": "Apenas consultas SELECT sao permitidas."})
    try:
        rows = conn.execute(query).fetchall()
        result = [dict(row) for row in rows]
        return json.dumps(result, ensure_ascii=False)
    except Exception as exc:
        return json.dumps({"error": str(exc)})


# Operadores matematicos permitidos na calculadora segura
_ALLOWED_OPS = {
    ast.Add:  operator.add,
    ast.Sub:  operator.sub,
    ast.Mult: operator.mul,
    ast.Div:  operator.truediv,
    ast.Pow:  operator.pow,
    ast.USub: operator.neg,
}


def _compute_ast_node(node):
    """Percorre recursivamente um no AST e computa o valor usando apenas operacoes matematicas."""
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError(f"Tipo nao permitido: {type(node.value)}")
    if isinstance(node, ast.BinOp):
        op_fn = _ALLOWED_OPS.get(type(node.op))
        if op_fn is None:
            raise ValueError(f"Operador nao permitido: {type(node.op).__name__}")
        return op_fn(_compute_ast_node(node.left), _compute_ast_node(node.right))
    if isinstance(node, ast.UnaryOp):
        op_fn = _ALLOWED_OPS.get(type(node.op))
        if op_fn is None:
            raise ValueError(f"Operador nao permitido: {type(node.op).__name__}")
        return op_fn(_compute_ast_node(node.operand))
    raise ValueError(f"No AST nao suportado: {type(node).__name__}")


def calculate(expression: str) -> str:
    """Avalia uma expressao matematica de forma segura via AST (modulo ast do Python)."""
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        result = _compute_ast_node(tree.body)
        return json.dumps({"result": result, "expression": expression.strip()})
    except Exception as exc:
        return json.dumps({"error": str(exc), "expression": expression.strip()})


# Registro das tools para despacho pelo nome
TOOL_REGISTRY = {
    "search_database": search_database,
    "calculate": calculate,
}

console.print("[green]Tools implementadas:[/green] search_database, calculate")

In [ ]:
# Declaracoes de tools no formato da API do Gemini (function calling)

TOOLS = [
    {
        "function_declarations": [
            {
                "name": "search_database",
                "description": "Executa consulta SQL SELECT no banco de vendas. "
                               "A tabela chama-se 'vendas' e tem as colunas: "
                               "id, produto, regiao, trimestre, ano, receita, unidades.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "Consulta SQL SELECT valida."
                        }
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "calculate",
                "description": "Avalia uma expressao matematica e retorna o resultado numerico. "
                               "Use para calcular porcentagens, diferencas, somas, etc.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Expressao matematica valida, ex: '(145000 - 138000) / 138000 * 100'."
                        }
                    },
                    "required": ["expression"]
                }
            }
        ]
    }
]

console.print(Panel(
    Syntax(json.dumps(TOOLS, indent=2, ensure_ascii=False), "json", theme="monokai"),
    title="Tool declarations enviadas ao Gemini",
    border_style="cyan"
))

## Passo 1: Primeira Chamada ao LLM

Enviamos a mensagem do usuario junto com as declaracoes de tools. O modelo vai receber a pergunta e decidir que precisa chamar uma tool antes de responder.

In [ ]:
# Primeira chamada — o LLM decide chamar uma tool

USER_MESSAGE = "Compare a receita do Q2 e Q3 de 2024 e calcule a diferenca percentual"

contents_step1 = [
    {"role": "user", "parts": [{"text": USER_MESSAGE}]}
]

response_step1 = client.models.generate_content(
    model=MODEL,
    contents=contents_step1,
    config={"tools": TOOLS}
)

console.print(Panel(f"[bold]Mensagem do usuario:[/bold] {USER_MESSAGE}",
                    border_style="white"))

# Inspecionar o que o LLM retornou
parts = response_step1.candidates[0].content.parts
for i, part in enumerate(parts):
    if hasattr(part, "function_call") and part.function_call:
        fc = part.function_call
        console.print(Panel(
            f"[yellow]Nome:[/yellow] {fc.name}\n"
            f"[yellow]Argumentos:[/yellow] {dict(fc.args)}",
            title=f"[bold yellow]Tool Call detectada (part {i})[/bold yellow]",
            border_style="yellow"
        ))
    elif hasattr(part, "text") and part.text:
        console.print(Panel(
            part.text,
            title=f"[bold green]Texto (part {i})[/bold green]",
            border_style="green"
        ))

## Passo 2: Executar a Tool

Agora que sabemos qual tool o modelo quer chamar e com quais argumentos, executamos a funcao Python correspondente e capturamos o resultado.

In [ ]:
# Executar a tool que o LLM solicitou

def execute_tool(function_call) -> dict:
    """Despacha a chamada de tool para a funcao Python correspondente."""
    name = function_call.name
    args = dict(function_call.args)

    if name not in TOOL_REGISTRY:
        return {"error": f"Tool desconhecida: {name}"}

    raw_result = TOOL_REGISTRY[name](**args)
    return json.loads(raw_result)


# Executar a primeira tool call do Step 1
first_part = next(
    (p for p in parts if hasattr(p, "function_call") and p.function_call),
    None
)

if first_part:
    tool_result = execute_tool(first_part.function_call)
    console.print(Panel(
        Syntax(json.dumps(tool_result, indent=2, ensure_ascii=False), "json", theme="monokai"),
        title=f"[bold cyan]Resultado de '{first_part.function_call.name}'[/bold cyan]",
        border_style="cyan"
    ))
else:
    console.print("[red]Nenhuma tool call encontrada no Step 1.[/red]")

## Passo 3: Enviar o Resultado de Volta ao LLM

O resultado da tool deve ser adicionado ao historico de conversa no formato que o Gemini espera: uma mensagem com `role=model` contendo o `function_call`, seguida de uma mensagem com `role=user` contendo o `function_response`.

Depois enviamos o historico atualizado para o LLM, que pode chamar outra tool ou dar a resposta final.

In [ ]:
# Construir historico com o resultado da tool e fazer a segunda chamada

# O historico agora tem: user_message → model_response (com function_call) → tool_result
contents_step3 = [
    # Mensagem original do usuario
    {"role": "user", "parts": [{"text": USER_MESSAGE}]},
    # Resposta do modelo (contendo o function_call)
    response_step1.candidates[0].content,
    # Resultado da tool (enviado como role=user com function_response)
    {
        "role": "user",
        "parts": [{
            "function_response": {
                "name": first_part.function_call.name,
                "response": tool_result
            }
        }]
    }
]

response_step3 = client.models.generate_content(
    model=MODEL,
    contents=contents_step3,
    config={"tools": TOOLS}
)

console.print(f"[bold]Historico enviado:[/bold] {len(contents_step3)} mensagens")

parts_step3 = response_step3.candidates[0].content.parts
for i, part in enumerate(parts_step3):
    if hasattr(part, "function_call") and part.function_call:
        fc = part.function_call
        console.print(Panel(
            f"[yellow]Nome:[/yellow] {fc.name}\n"
            f"[yellow]Argumentos:[/yellow] {dict(fc.args)}",
            title=f"[bold yellow]Nova Tool Call (part {i})[/bold yellow]",
            border_style="yellow"
        ))
    elif hasattr(part, "text") and part.text:
        console.print(Panel(
            part.text,
            title=f"[bold green]Resposta do LLM (part {i})[/bold green]",
            border_style="green"
        ))

## O Loop Completo

Os tres passos acima sao sempre os mesmos. Basta envolver tudo em um loop `for step in range(max_steps)` que continua ate que o LLM retorne texto em vez de uma tool call.

In [ ]:
# Loop ReAct completo — implementacao manual sem framework

def react_loop(user_message: str, max_steps: int = 10) -> str:
    """Loop ReAct manual — sem framework.

    Retorna a resposta final do LLM ou uma mensagem de limite atingido.
    """
    contents = [
        {"role": "user", "parts": [{"text": user_message}]}
    ]

    console.rule("[bold blue]Inicio do Loop ReAct[/bold blue]")
    console.print(Panel(f"[bold]Pergunta:[/bold] {user_message}", border_style="white"))

    for step in range(1, max_steps + 1):
        console.rule(f"[dim]Step {step}[/dim]")

        # 1. Chamar o LLM com o historico atual
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config={"tools": TOOLS}
        )

        model_content = response.candidates[0].content
        parts = model_content.parts

        # Adicionar a resposta do modelo ao historico
        contents.append(model_content)

        # 2. Inspecionar as parts da resposta
        tool_results = []
        final_text = None

        for part in parts:
            if hasattr(part, "function_call") and part.function_call:
                fc = part.function_call
                console.print(
                    f"  [yellow]TOOL CALL[/yellow] → [bold]{fc.name}[/bold]({dict(fc.args)})"
                )

                # 3. Executar a tool
                result = execute_tool(fc)
                console.print(
                    f"  [cyan]RESULTADO[/cyan] → {json.dumps(result, ensure_ascii=False)[:200]}"
                )

                tool_results.append({
                    "name": fc.name,
                    "response": result
                })

            elif hasattr(part, "text") and part.text:
                final_text = part.text

        # 4. Se houve tool calls, adicionar os resultados ao historico
        if tool_results:
            function_response_parts = [
                {"function_response": tr}
                for tr in tool_results
            ]
            contents.append({
                "role": "user",
                "parts": function_response_parts
            })
            # Continuar o loop — o modelo ainda tem que processar os resultados
            continue

        # 5. Se nao houve tool calls e ha texto, e a resposta final
        if final_text:
            console.print(Panel(
                final_text,
                title="[bold green]Resposta Final[/bold green]",
                border_style="green"
            ))
            console.print(f"[dim]Loop concluido em {step} step(s).[/dim]")
            return final_text

    # Limite de steps atingido
    msg = f"Limite de {max_steps} steps atingido sem resposta final."
    console.print(f"[red]{msg}[/red]")
    return msg

In [ ]:
# Executar o loop com a pergunta completa

resposta = react_loop(
    "Compare a receita do Q2 e Q3 de 2024 e calcule a diferenca percentual"
)

## Comparando com LangGraph

Abaixo, uma comparacao entre a implementacao manual e o equivalente usando LangGraph.

| Aspecto | ReAct Manual (este notebook) | LangGraph |
|---|---|---|
| **Loop principal** | `for step in range(max_steps)` escrito por voce | `graph.compile()` + `graph.stream()` |
| **Gerenciamento de estado** | Lista `contents` manipulada manualmente | `MessagesState` com reducers tipados |
| **Execucao de tools** | `hasattr(part, 'function_call')` + despacho manual | `ToolNode` cuida automaticamente |
| **Decisao de parar** | Checar se ha texto sem function_call | Edge condicional `should_continue` |
| **Streaming** | Nao incluso (requer stream separado) | `graph.stream()` nativo por no |
| **Checkpointing** | Nao incluso | `MemorySaver` / `SqliteSaver` integrado |
| **Linhas de codigo** | ~50 linhas | ~30 linhas (mais setup do grafo) |
| **Debugabilidade** | Total — voce ve cada byte | Requer LangSmith ou inspecao de eventos |

```python
# ReAct Manual (este notebook)
for step in range(max_steps):
    response = client.models.generate_content(model, contents, config={"tools": tools})
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'function_call') and part.function_call:
            result = execute_tool(part.function_call)
            contents.append({"role": "user", "parts": [{"function_response": ...}]})
        elif hasattr(part, 'text') and part.text:
            return part.text

# LangGraph equivalente
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_conditional_edges("agent", should_continue)
app = graph.compile()
result = app.invoke({"messages": [HumanMessage(content=user_message)]})
```

## O que LangGraph Adiciona

Implementar o loop manualmente e valioso para entender o que acontece por baixo, mas na producao LangGraph oferece funcionalidades trabalhosas de construir do zero:

**Estrutura de Grafo**
- Modela o fluxo do agente como um grafo dirigido com nos e arestas.
- Facil de visualizar, testar e modificar partes sem reescrever tudo.

**Estado Tipado**
- O estado do agente tem schema definido (`TypedDict`, Pydantic).
- Reducers controlam como mensagens se acumulam — sem manipulacao manual de listas.

**Streaming Nativo**
- `graph.stream()` emite eventos por no: voce ve exatamente quando o agente pensou, quando chamou a tool, quando recebeu o resultado.
- Com a implementacao manual precisaria de API de streaming separada e mais codigo.

**Checkpointing e Persistencia**
- `MemorySaver` / `SqliteSaver` salvam o estado apos cada step.
- Se o processo cair no step 4 de 7, a execucao pode ser retomada do ponto exato.

**Human-in-the-Loop**
- `interrupt_before=["tools"]` pausa o grafo antes de executar qualquer tool.
- Um humano pode revisar e aprovar (ou rejeitar) a acao antes de ela acontecer.

**Multi-agente**
- Nos do grafo podem ser outros grafos (subgrafos).
- Orquestra varios agentes especializados com passagem de estado entre eles.

## Tente Voce

Agora que voce entende o loop por dentro, aqui estao tres exercicios para aprofundar:

**Exercicio 1 — Terceira tool: `top_produtos`**

Adicione uma nova tool chamada `top_produtos` que recebe um parametro `n` (inteiro) e retorna os `n` produtos com maior receita total no banco. Declare-a em `TOOLS` e adicione-a em `TOOL_REGISTRY`. Teste com a pergunta: "Qual o produto mais vendido em 2024?".

**Exercicio 2 — Limite de steps com log detalhado**

Modifique `react_loop` para logar com Rich, a cada step:
- Numero do step e numero de mensagens no historico.
- Nome da tool chamada (se houver) e tamanho do resultado em bytes.
- Tempo decorrido desde o inicio do loop (use `time.perf_counter`).

Teste com `max_steps=2` numa pergunta complexa e observe o comportamento quando o limite e atingido.

**Exercicio 3 — Suporte a multiplas tool calls por step**

O Gemini pode retornar mais de uma `function_call` na mesma resposta (em `parts` diferentes). O loop atual ja suporta isso — mas verifique: envie uma pergunta que force o modelo a chamar `search_database` e `calculate` ao mesmo tempo (hint: pergunte sobre dois trimestres diferentes em uma unica frase). Observe se o modelo paralleliza as chamadas ou as encadeia sequencialmente.